# Guided Project: Data Wrangling with Pandas & NumPy for S3 Pipelines

---

## 🌟 Overview: What is the purpose of this Lab?

Welcome to Lab 2 of your Data Engineering journey! If you've ever wondered how companies like Spotify or Uber take messy, raw employee or transactional data and turn it into clean, analytics-ready datasets stored in the cloud — this is exactly how they do it.

**The Purpose of this Lab:**
Raw data is almost never clean. In production pipelines, data arrives with missing values, inconsistent casing, duplicate records, wrong data types, and outliers. This lab takes you through the complete **Bronze to Silver** transformation — the industry-standard first step in a Medallion Architecture data lake.

By the end of this project, you will have built a Python data wrangling pipeline that uploads raw data to Amazon S3, profiles it for quality issues, applies systematic cleaning using Pandas and NumPy, exports the cleaned result to Parquet format, and pushes it back to S3 — all from a Jupyter Notebook.

### Learning Path:
1.  **Cloud Storage Setup:** Create an Amazon S3 bucket to hold raw and cleaned data.
2.  **Authentication:** Configure AWS CLI credentials so Boto3 can talk to S3.
3.  **Ingestion:** Upload the raw CSV to S3 and download it into Pandas.
4.  **Data Profiling:** Detect nulls, duplicates, type issues, and value inconsistencies.
5.  **NumPy Analysis:** Compute statistical summaries and detect salary outliers using IQR.
6.  **Data Cleaning:** Apply a systematic series of Pandas transformations.
7.  **Export & Validate:** Write cleaned data to Parquet and verify the round-trip from S3.

---

## 📊 Dataset / Knowledge Source Used

File: `~/Desktop/Project/employees_raw.csv` _(Already provided in your workspace)_

Content:
- ~1,000 employee records across departments including Engineering, Sales, HR, Legal, Finance, and more.
- Intentionally contains real-world data quality issues: missing values, inconsistent casing, leading/trailing whitespace, duplicate rows, invalid date formats, and salary outliers.

This file will be:
- Uploaded programmatically to an **Amazon S3** bucket you create via the Console.
- Downloaded back into this Jupyter Notebook for profiling and wrangling.
- Exported as **Parquet + Snappy** and stored back in S3.

---

## 🛠️ Prerequisites & AWS Setup

Ensure the following are available in your workspace:
- An **AWS account** with IAM permissions to use S3.
- **VS Code** installed with the Jupyter extension pre-installed.
- The local data file at `~/Desktop/Project/employees_raw.csv`.

---

## Configure AWS Credentials

First, we need to log in to the AWS Web Console to set up our destination.

- Login to AWS Console:
  - Click on the `Lab Access` icon on the desktop.

![Images](images/lab-image1.png)

  - Click on `Access Lab` and using the given credentials login to AWS Console.

![Images](images/lab-image2.png)

  - Once logged in, set the region to `us-east-1`. All resources must be created in **US-EAST-1 (N. Virginia)**.
  - Locate the region selector at the top-right corner of the console (next to your Account Name).

![Images](images/lab-image3.png)

  - Choose **us-east-1 (N. Virginia)**.

![Images](images/lab-image4.png)

---

# Activity 1: Create Amazon S3 Bucket via AWS Console

You will create the target S3 bucket manually through the console. This bucket will hold both the raw input CSV (Bronze zone) and the cleaned Parquet output (Silver zone).

1. In the AWS Management Console search bar, type **S3** and select it from the services menu.

![Images](images/1search_click_s3.png)

2. Click the **Create bucket** button.

![Images](images/2create_bucket.png)

3. **Bucket Name Configuration:**
   - Provide a globally unique name using the following format: `data-wrangling-lab-<USER_INPUT>`
   - _Replace `<USER_INPUT>` with your unique identifier. For example: `data-wrangling-lab-bucket`._
4. **AWS Region:** Ensure it is set to **us-east-1 (N. Virginia)**.

![Images](images/3bucket_name_region.png)

5. Leave all other settings as default. **Block all public access** must remain checked.
6. Scroll to the bottom and click **Create bucket**.

![Images](images/4bucket_name_region.png)

Inside the bucket, create two folders:

| Folder | Purpose |
|---|---|
| `raw/` | Stores the original unmodified CSV — the Bronze zone |
| `cleaned/` | Stores the Parquet output after wrangling — the Silver zone |

#### Create the `raw/` Folder
- Click on **Create folder**

![Images](images/bucket_folders_1.png)

- Enter folder name: `raw` and click **Create folder**.

![Images](images/bucket_folders.png)

#### Create the `cleaned/` Folder
- Click on **Create folder** again.

![Images](images/bucket_folders_2.png)

- Enter folder name: `cleaned` and click **Create folder**.

![Images](images/bucket_folders1.png)

---

# Verify AWS Configuration

AWS is already configured in this environment. Let's verify the configuration is active before we proceed.

### Step 1: Open Terminal in VS Code
- Press **Ctrl + `** (backtick key, below ESC) to open the integrated terminal.

![Images](images/terminal.png)

### Step 2: Verify AWS Configuration
Run the command below in your VS Code terminal to confirm AWS knows who you are.

In [ ]:
# 👇 DO NOT RUN THIS CELL — copy the command below and run it in your VS Code Terminal 👇
# aws sts get-caller-identity

---

If configured correctly, you will see a JSON response in your terminal containing your `UserId`, `Account`, and `Arn`.

![Images](images/terminal2.png)

> ⚠️ **If the command above returned an error** (e.g. `Unable to locate credentials` or `command not found`), AWS is **not yet configured**. Complete **Step 3** below before continuing.

### Step 3: Set Up AWS CLI Access

> ✅ **Skip this step** if `aws sts get-caller-identity` in Step 2 already returned your `UserId`, `Account`, and `Arn` successfully.


To configure your credentials, you need to run the AWS configure command in your VS Code terminal.
Since traditional copy-pasting into web terminals can be tricky, please carefully select and copy the text from the code cell below.

In [ ]:
# 👇 Select and copy the command below, then paste it into your VS Code terminal 👇
# aws configure

When prompted in the terminal, enter your credentials exactly like this:
```text
AWS Access Key ID [None]: <Paste your AccessKeyID here>
AWS Secret Access Key [None]: <Paste your SecretAccessKey here>
Default region name [None]: us-east-1
Default output format [None]: json
```

![Images](images/aws_configure.png)

---

# Activity 2: Import Libraries & Connect to S3

Now that AWS credentials are configured, we switch fully to the Jupyter Notebook. Every step from here runs as a code cell.

> 💡 Ensure the kernel selector in the top-right shows **Python 3** (or the pre-configured kernel for this lab).

### Step 2.1: Import Required Libraries

💡 **What this code does:**
- `pandas` — loads, inspects, and transforms tabular data. This is our primary wrangling tool throughout the lab.
- `numpy` — provides fast numerical operations, statistical functions, and array handling for deeper data profiling.
- `boto3` — the official AWS SDK for Python. It lets our script talk directly to S3 without using the browser.
- `io` — creates in-memory file buffers so we can read and write S3 objects without saving anything to local disk.
- `botocore.exceptions` — gives us the `ClientError` class to catch AWS API errors and print meaningful messages.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
import pandas as pd                          # DataFrame operations — our main wrangling tool
import numpy as np                           # Numerical arrays and statistical functions
import boto3                                 # AWS SDK — connects Python to S3
import io                                    # In-memory byte/string stream handling
from botocore.exceptions import ClientError  # Catches AWS API errors cleanly

print("Libraries imported successfully!")
print(f"   pandas version : {pd.__version__}")
print(f"   numpy  version : {np.__version__}")
print(f"   boto3  version : {boto3.__version__}")

![Images](images/execution1.png)

> 💡 **TODO**
> Complete the function below to import the `os` module and return the current working directory.
>
> Please ensure you complete and execute the code cell below, as your results will be validated and automatically reflected in your **Practice Progress Summary** to track your learning milestones.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Complete the code with the `# TODO` comments. Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.


In [ ]:
def get_working_directory():
    result = None

    # code starts here

    # TODO: Import the os module and use it to get the current working directory
    # TODO: Store the path in 'result'

    # code ends here

    return result

working_dir = get_working_directory()
print("Current Working Directory:", working_dir)

> 🎯 **Try Out**
> In the blank cell below, import the `csv` module and the `json` module. Then use Python's built-in `dir()` function to print all available methods in the `csv` module. This is a useful trick for discovering what tools a library provides.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!


### Step 2.2: Set Your Bucket Name & Initialize the S3 Client

💡 **What this code does:**
- `BUCKET_NAME` holds the name of the S3 bucket you created in Activity 1. Replace `<USER_INPUT>` with your unique identifier.
- `RAW_KEY` and `CLEANED_KEY` define the S3 object paths — these act like folder + filename inside your bucket.
- `boto3.client("s3")` creates a connection object to the S3 service. Every upload, download, and list operation in this lab goes through this client.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Edit `<USER_INPUT>` to match the identifier you used in Activity 1, then press **Shift + Enter**.

In [ ]:
# ============================================================
# TODO: Replace <USER_INPUT> with your unique identifier
# Example: BUCKET_NAME = "data-wrangling-lab-<USER_INPUT>"
# ============================================================
BUCKET_NAME = "data-wrangling-lab-<USER_INPUT>"  # <-- EDIT THIS

RAW_KEY     = "raw/employees_raw.csv"             # S3 path for the raw CSV (Bronze zone)
CLEANED_KEY = "cleaned/employees_cleaned.parquet" # S3 path for the Parquet output (Silver zone)
LOCAL_CSV   = "employees_raw.csv"                 # Local filename available in your workspace

# Create an S3 client — this object handles all communication with AWS S3
s3 = boto3.client("s3", region_name="us-east-1")

print(f"S3 client ready")
print(f"   Target bucket  : {BUCKET_NAME}")
print(f"   Raw input path : s3://{BUCKET_NAME}/{RAW_KEY}")
print(f"   Cleaned output : s3://{BUCKET_NAME}/{CLEANED_KEY}")

![Images](images/execution2.png)

> 💡 **TODO**
> Complete the function below to build and return a full S3 URI string using `bucket_name` and `key`.
>
> Please ensure you complete and execute the code cell below, as your results will be validated and automatically reflected in your **Practice Progress Summary** to track your learning milestones.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Complete the code with the `# TODO` comments. Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.


In [ ]:
def build_s3_uri(bucket_name, key):
    result = None

    # code starts here

    # TODO: Combine bucket_name and key into an S3 URI (format: s3://bucket/key)
    # TODO: Store the URI string in 'result'

    # code ends here

    return result

s3_uri = build_s3_uri(BUCKET_NAME, RAW_KEY)
print("S3 URI:", s3_uri)

> 🎯 **Try Out**
> In the blank cell below, define a Python dictionary called `s3_paths` with three keys: `"bucket"`, `"raw"`, and `"cleaned"`. Assign each key the appropriate value from the variables defined above. Then loop through the dictionary and print each key-value pair on its own line.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!


---

# Activity 3: Upload the Raw CSV to S3

Before we can load data from S3, we first upload our local CSV file to the `raw/` folder in the bucket. This is the **Bronze ingestion step** — loading raw, unmodified data into cloud storage.

### Step 3.1: Upload the Local File to S3

💡 **What this code does:**
- `s3.upload_file()` transfers the local `employees_raw.csv` from your workspace to the S3 path `raw/employees_raw.csv`.
- The `try/except` block catches any errors (wrong bucket name, missing permissions) and prints a clear message instead of crashing the notebook.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Upload the local CSV file to the S3 raw/ prefix
try:
    s3.upload_file(LOCAL_CSV, BUCKET_NAME, RAW_KEY)
    print(f"Uploaded  ->  s3://{BUCKET_NAME}/{RAW_KEY}")
except ClientError as e:
    print(f"Upload failed: {e}")

![Images](images/execution3.png)

### Step 3.2: Download the File from S3 into a Pandas DataFrame

💡 **What this code does:**
- `s3.get_object()` downloads the file from S3 — the response contains the file content as raw bytes.
- `io.BytesIO()` wraps those bytes in an in-memory buffer so `pd.read_csv()` can read it exactly like a local file — no disk write needed.
- `.head()` shows the first 5 rows so we can confirm the data looks right before going further.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Download from S3 — response body contains the raw file bytes
response  = s3.get_object(Bucket=BUCKET_NAME, Key=RAW_KEY)
csv_bytes = response["Body"].read()   # Read all bytes from the S3 response stream

# Load bytes into Pandas using an in-memory buffer (no local file saved)
df_raw = pd.read_csv(io.BytesIO(csv_bytes))

print(f"Data loaded from S3 | Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print()
df_raw.head()

![Images](images/execution4.png)

> 🎯 **Try Out**
> In the blank cell below, write code to print how many rows and columns `df_raw` has by accessing `df_raw.shape`. Then print only the rows where the `department` column equals `"Finance"` using boolean indexing (`df_raw[df_raw["department"] == "Finance"]`). Show only the first 3 results with `.head(3)`.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!


---

# Activity 4: Data Profiling — Understand the Dataset

Before making any changes, a good Data Engineer **profiles** the data first — understanding its shape, types, and quality issues. This prevents surprises and guides every cleaning decision.

### Step 4.1: Inspect Shape and Column Data Types

💡 **What this code does:**
- `.shape` returns `(rows, columns)` — a quick sanity check on how much data we are working with.
- `.dtypes` shows the data type Pandas assigned to each column after loading. This immediately reveals problems — for example, if a numeric column like `salary` was loaded as `object` (string), we know there are mixed or bad values in that column.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Overall size of the dataset — rows and columns
print("=== Shape ===")
print(f"Rows: {df_raw.shape[0]},  Columns: {df_raw.shape[1]}")
print()

![Images](images/execution5.png)

In [ ]:
# Data type of each column — tells us what Pandas inferred automatically on load
print("=== Column Data Types ===")
print(df_raw.dtypes)

![Images](images/execution6.png)

### Step 4.2: Check for Missing Values — `isna()`

💡 **What this code does:**
- `.isnull()` (same as `.isna()`) returns a boolean DataFrame where `True` marks every null/missing cell.
- `.sum()` counts the number of `True` values per column — giving us the **null count** per column.
- Dividing by the total row count and multiplying by 100 gives the **null rate %** — this tells us how severely each column is affected and whether we should drop or fill.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Count nulls per column
missing     = df_raw.isnull().sum()

# Calculate what percentage of each column is missing
missing_pct = (missing / len(df_raw) * 100).round(2)

# Combine into a readable summary table
missing_report = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})

print("=== Missing Values Report ===")
print(missing_report)

![Images](images/execution7.png)

### Step 4.3: Detect Duplicate Rows — `duplicated()`

💡 **What this code does:**
- `.duplicated()` marks rows that are exact copies of a previous row as `True`.
- `.sum()` counts how many such duplicate rows exist in the full dataset.
- Duplicates commonly appear when data is received from multiple sources or when an ingestion job runs more than once without deduplication.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Count rows that are exact copies of a previous row
duplicate_count = df_raw.duplicated().sum()

print("=== Duplicate Rows ===")
print(f"Exact duplicate rows found: {duplicate_count}")

![Images](images/execution8.png)

### Step 4.4: Explore Categorical Column Values — `unique()`

💡 **What this code does:**
- `.unique()` returns all distinct values in a column — this is how we spot inconsistent casing, typos, or unexpected values.
- For example, `"Engineering"`, `"engineering"`, and `"ENGINEERING"` are three separate values in Pandas but should be one. Seeing this here tells us exactly what cleaning is needed.
- `dropna()` is chained before `unique()` to exclude nulls from the display — we handle nulls separately.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Check unique values in the department column — look for casing inconsistencies
print("=== Department — Unique Values ===")
print(sorted(df_raw["department"].dropna().unique()))
print()

![Images](images/execution9.png)



In [ ]:
# Check unique values in the status column
print("=== Status — Unique Values ===")
print(df_raw["status"].dropna().unique())

![Images](images/execution10.png)


> 💡 **TODO**
> Complete the function below to profile the DataFrame for missing values — return a dictionary mapping each column name to its null count.
>
> Please ensure you complete and execute the code cell below, as your results will be validated and automatically reflected in your **Practice Progress Summary** to track your learning milestones.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>

> **⚡ How to run the code:** Complete the code with the `# TODO` comments. Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
def get_null_counts(df):
    result = None

    # code starts here

    # TODO: Use isnull().sum() to compute null counts per column
    # TODO: Convert the result to a dictionary and store in 'result'

    # code ends here

    return result

null_counts = get_null_counts(df_raw)
print("Null counts per column:", null_counts)

### Challenge 1: Profile the `salary` Column with Pandas

Now it's your turn! Write code below to profile the `salary` column using Pandas:
1. Print the **number of missing values** in `salary`.
2. Print the **minimum**, **maximum**, and **mean** salary using Pandas (`.min()`, `.max()`, `.mean()`).
3. Use `.value_counts(bins=4)` to show how salaries are distributed across 4 equal-width buckets.

*Major Hint: Access the column with `df_raw["salary"]`. Chain `.dropna()` before stat functions to skip missing values automatically.*

**⚡ How to run the code:**
Write your solution below, then press **Shift + Enter**. *(Solution at the bottom of this notebook!)*

In [ ]:
# Challenge 1: Write your solution here!

# Step 1: Count missing values in salary

# Step 2: Print min, max, and mean using Pandas

# Step 3: Show salary distribution across 4 buckets


---

# Activity 5: NumPy Statistical Profiling

Pandas `.describe()` gives a summary, but NumPy lets us go deeper — computing IQR bounds and detecting salary outliers that would distort our analysis if left unchecked.

### Step 5.1: Compute Salary Statistics with NumPy

💡 **What this code does:**
- `.dropna().values` extracts the `salary` column as a raw NumPy array — dropping nulls so they don't affect the calculations.
- `np.mean`, `np.median`, `np.std`, `np.min`, `np.max` are NumPy functions that operate directly on the array.
- Working on a NumPy array is faster than Pandas for large numerical datasets because NumPy uses optimized C-level vectorized operations under the hood.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Extract salary column as a NumPy array — drop nulls first so stats are accurate
salary_array = df_raw["salary"].dropna().values

print("=== Salary Statistics (NumPy) ===")
print(f"Count  : {len(salary_array):,}")
print(f"Mean   : {np.mean(salary_array):,.2f}")
print(f"Median : {np.median(salary_array):,.2f}")
print(f"Std Dev: {np.std(salary_array):,.2f}")
print(f"Min    : {np.min(salary_array):,.2f}")
print(f"Max    : {np.max(salary_array):,.2f}")

![Images](images/execution11.png)

> 💡 **TODO**
> Complete the function below to compute basic NumPy statistics — mean, median, and standard deviation — for a given numeric array.
>
> Please ensure you complete and execute the code cell below, as your results will be validated and automatically reflected in your **Practice Progress Summary** to track your learning milestones.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Complete the code with the `# TODO` comments. Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
def compute_numpy_stats(array):
    mean = None
    median = None
    std = None

    # code starts here

    # TODO: Use np.mean() to compute the mean of the array and store in 'mean'
    # TODO: Use np.median() to compute the median of the array and store in 'median'
    # TODO: Use np.std() to compute the standard deviation and store in 'std'

    # code ends here

    return (mean, median, std)

stats = compute_numpy_stats(salary_array)
print(f"Mean   : {stats[0]:,.2f}")
print(f"Median : {stats[1]:,.2f}")
print(f"Std Dev: {stats[2]:,.2f}")

### Step 5.2: IQR-Based Outlier Detection

💡 **What this code does:**
- **IQR (Interquartile Range)** is the spread of the middle 50% of values — computed as Q3 minus Q1. It is resistant to extreme values, unlike the standard deviation.
- Any salary below `Q1 - 1.5 x IQR` or above `Q3 + 1.5 x IQR` is flagged as a statistical outlier.
- `np.percentile()` computes any percentile directly from the array — here we use it for the 25th and 75th.
- Outliers here represent salaries that are abnormally high or low — they could indicate data entry errors or edge cases like executives or interns.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Compute the 25th and 75th percentiles from the salary array
Q1  = np.percentile(salary_array, 25)
Q3  = np.percentile(salary_array, 75)
IQR = Q3 - Q1    # Middle 50% spread — resistant to outliers

# Any value outside these bounds is a statistical outlier (1.5 x IQR rule)
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 (25th pct) : {Q1:,.2f}")
print(f"Q3 (75th pct) : {Q3:,.2f}")
print(f"IQR           : {IQR:,.2f}")
print(f"Lower bound   : {lower_bound:,.0f}")
print(f"Upper bound   : {upper_bound:,.0f}")
print()

![Images](images/execution12.png)

In [ ]:
# Identify salary values that fall outside the computed bounds
outliers = salary_array[(salary_array < lower_bound) | (salary_array > upper_bound)]

print(f"Outlier count : {len(outliers)}")
print(f"Outlier values: {sorted(outliers)}")

![Images](images/execution13.png)

> 💡 **TODO**
> Complete the function below to count how many salaries fall above the `upper_bound` and how many fall below the `lower_bound`.
>
> Please ensure you complete and execute the code cell below, as your results will be validated and automatically reflected in your **Practice Progress Summary** to track your learning milestones.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Complete the code with the `# TODO` comments. Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.


In [ ]:
def count_outliers(salary_array, upper_bound, lower_bound):
    high_count = None
    low_count = None

    # code starts here

    # TODO: Count salaries above upper_bound and store in 'high_count'
    # TODO: Count salaries below lower_bound and store in 'low_count'

    # code ends here

    return (high_count, low_count)

high, low = count_outliers(salary_array, upper_bound, lower_bound)
print(f"High outliers (above upper bound): {high}")
print(f"Low outliers  (below lower bound): {low}")

> 🎯 **Try Out**
> In the blank cell below, use NumPy to compute the **90th percentile** salary using `np.percentile(salary_array, 90)`. Then count how many employees earn above this threshold and print that count as a percentage of the total salary array length.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!


---

### Challenge 2: NumPy Analysis on Another Column

Repeat the NumPy statistical analysis, but this time on the `employee_id` column:
1. Extract `employee_id` as a NumPy array (drop nulls first).
2. Print the **count**, **min**, **max**, and **mean** using NumPy functions.
3. Use `np.unique()` to find how many **unique employee IDs** exist — compare this to the total row count. What does a mismatch tell you?

*Major Hint: Use `df_raw["employee_id"].dropna().values` to get the array. `len(np.unique(array))` gives you the unique count.*

**⚡ How to run the code:**
Write your solution below, then press **Shift + Enter**. *(Solution at the bottom!)*

In [ ]:
# Challenge 2: Write your solution here!

# Step 1: Extract employee_id as a NumPy array

# Step 2: Print count, min, max, mean

# Step 3: Count unique IDs and compare to total rows


---

# Activity 6: Data Cleaning Transformations

Now that we understand the data, we apply a systematic series of cleaning steps. We always work on a **copy** of the raw DataFrame so the original stays intact for comparison and audit.

### Step 6.1: Create a Working Copy

💡 **What this code does:**
- `.copy()` creates a completely independent copy of `df_raw`. Any changes to `df` will not affect `df_raw`.
- This is a best practice in data engineering — always preserve raw data in its original state so you can compare before and after, and re-run cleaning steps from scratch if needed.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Always work on a copy — never modify the raw DataFrame directly
df = df_raw.copy()

print(f"Working copy created: {df.shape[0]} rows x {df.shape[1]} columns")

![Images](images/execution14.png)

### Step 6.2: Remove Duplicate Rows — `drop_duplicates()`

💡 **What this code does:**
- `.drop_duplicates()` removes rows that are exact copies of a previous row, keeping only the first occurrence of each.
- We capture the row count before and after so we can confirm how many duplicates were removed.
- Duplicates inflate aggregations (totals, counts, averages) and must be eliminated before analysis.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
rows_before = len(df)

# Remove all rows that are exact duplicates — keep the first occurrence
df = df.drop_duplicates()

print(f"[1] Duplicate rows removed : {rows_before - len(df)}")
print(f"    Rows before: {rows_before}  ->  Rows after: {len(df)}")

![Images](images/execution15.png)

### Step 6.3: Strip Whitespace from the `name` Column — `str.strip()`

💡 **What this code does:**
- `.str.strip()` removes any leading or trailing whitespace characters from every string value in the column.
- Without this, `"Alice "` and `"Alice"` are treated as two different names — breaking joins, filters, and groupBy operations silently.
- We verify the fix by checking how many names still have leading or trailing spaces after the operation.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Remove leading and trailing whitespace from every name
df["name"] = df["name"].str.strip()

# Confirm no names still carry extra spaces
spaces_remaining = (df["name"].dropna().str.startswith(" ").sum() +
                    df["name"].dropna().str.endswith(" ").sum())

print(f"[2] Whitespace stripped from name column")
print(f"    Names still with leading/trailing spaces: {spaces_remaining}")

![Images](images/execution16.png)

### Step 6.4: Standardize `department` to Title Case — `str.title()`

💡 **What this code does:**
- `.str.strip()` removes extra spaces first.
- `.str.title()` converts all strings to Title Case — so `"engineering"`, `"ENGINEERING"`, and `"Engineering"` all become `"Engineering"`.
- After this step, groupBy on `department` will correctly combine all rows for the same department instead of treating each casing variant as a separate group.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
print("Before — unique department values:")
print(sorted(df["department"].dropna().unique()))

# Strip whitespace first, then convert to Title Case
df["department"] = df["department"].str.strip().str.title()

print("\nAfter — unique department values:")
print(sorted(df["department"].dropna().unique()))
print(f"\n[3] Department standardised to Title Case")

![Images](images/execution17.png)

### Step 6.5: Standardize `status` to Lowercase — `str.lower()`

💡 **What this code does:**
- `.str.lower()` converts all status values to lowercase — so `"Active"`, `"ACTIVE"`, and `"active"` all become `"active"`.
- This is critical for any downstream system that filters on this field — a SQL `WHERE status = 'active'` would miss rows with `"Active"` or `"ACTIVE"` without this fix.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
print("Before — unique status values:")
print(df["status"].dropna().unique())

# Standardize all status values to lowercase
df["status"] = df["status"].str.strip().str.lower()

print("\nAfter — unique status values:")
print(sorted(df["status"].dropna().unique()))
print(f"\n[4] Status standardised to lowercase")

![Images](images/execution18.png)

### Step 6.6: Fill Missing `name` Values — `fillna()`

💡 **What this code does:**
- `.fillna("Unknown")` replaces every null value in the `name` column with the placeholder string `"Unknown"`.
- This keeps the row in the dataset — which is important when other columns in that row have valid data — while clearly flagging that the name was missing.
- We verify the fix by checking that no nulls remain in the column.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Fill missing name values with a readable placeholder
df["name"] = df["name"].fillna("Unknown")

print(f"[5] Missing names filled with 'Unknown'")
print(f"    Remaining nulls in name: {df['name'].isna().sum()}")

![Images](images/execution19.png)

> 🎯 **Try Out**
> In the blank cell below, write a line to fill missing values in the `department` column with the string `"Unassigned"`. Then print the count of remaining nulls to verify. Use the same pattern as Step 7.6.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!


### Step 6.7: Fill Missing `salary` with NumPy Median

💡 **What this code does:**
- `np.median()` is called on the clean (non-null) salary values to compute the median — the middle value of the sorted array.
- We use the **median** instead of the mean because the outliers we detected in Activity 6 would pull the mean higher, making it a less representative fill value.
- `.fillna(median_salary)` then replaces every null salary with this computed value.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Compute median salary using NumPy — robust to outliers unlike the mean
median_salary = np.median(df["salary"].dropna().values)

# Fill all null salaries with the computed median
df["salary"] = df["salary"].fillna(median_salary)

print(f"[6] Missing salary filled with NumPy median: {median_salary:,.2f}")
print(f"    Remaining nulls in salary: {df['salary'].isna().sum()}")

![Images](images/execution20.png)


### Step 6.8: Fix Data Type — Cast `salary` to Integer

💡 **What this code does:**
- After filling nulls, `salary` is stored as a float (because the median fill value may be a decimal). We cast it to `int` since salary values should be whole numbers.
- `.astype(int)` converts every value in the column to a Python integer.
- This ensures the column type is correct for downstream tools like Athena or Spark that expect integer salary fields.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
print(f"Before: salary dtype = {df['salary'].dtype}")

# Cast salary from float to integer
df["salary"] = df["salary"].astype(int)

print(f"After : salary dtype = {df['salary'].dtype}")
print(f"[7] Salary cast to int")

![Images](images/execution21.png)


### Step 6.9: Parse `hire_date` to a Proper Datetime — `pd.to_datetime()`

💡 **What this code does:**
- `hire_date` was loaded as a plain string (`object` dtype). `pd.to_datetime()` converts it to a proper `datetime64` type so Pandas understands it as a date.
- `errors="coerce"` tells Pandas to convert unparseable date strings to `NaT` (Not a Time) instead of raising an error and stopping.
- Once converted, date operations like filtering by year, computing tenure, or sorting by date all work correctly.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
print(f"Before: hire_date dtype = {df['hire_date'].dtype}")

# Parse the date strings — unparseable values become NaT instead of an error
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")

unparseable = df["hire_date"].isna().sum()
print(f"After : hire_date dtype = {df['hire_date'].dtype}")
print(f"[8] hire_date parsed | Unparseable rows set to NaT: {unparseable}")

![Images](images/execution22.png)


### Step 6.10: Fill Remaining NaT in `hire_date` with a Sentinel Date

💡 **What this code does:**
- Some `hire_date` values may still be `NaT` from the previous step (dates that were malformed and couldn't be parsed).
- We fill these with the sentinel date `1900-01-01` — a clear signal that the date is unknown, without dropping the row.
- `pd.Timestamp()` creates a proper datetime value compatible with the column's `datetime64` type.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Replace remaining NaT dates with a sentinel placeholder date
df["hire_date"] = df["hire_date"].fillna(pd.Timestamp("1900-01-01"))

print(f"[9] NaT hire_date filled with sentinel 1900-01-01")
print(f"    Remaining NaT in hire_date: {df['hire_date'].isna().sum()}")

![Images](images/execution23.png)

> 💡 **TODO**
> Complete the function below to compare the raw and cleaned DataFrames — return row counts and total null counts for both.
>
> Please ensure you complete and execute the code cell below, as your results will be validated and automatically reflected in your **Practice Progress Summary** to track your learning milestones.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>

> **⚡ How to run the code:** Complete the code with the `# TODO` comments. Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.


In [ ]:
def get_cleaning_summary(df_raw, df_clean):
    raw_rows = None
    clean_rows = None
    raw_nulls = None
    clean_nulls = None

    # code starts here

    # TODO: Get the row count for df_raw and df_clean and store in 'raw_rows' and 'clean_rows'
    # TODO: Get the total null count for df_raw and df_clean and store in 'raw_nulls' and 'clean_nulls'

    # code ends here

    return (raw_rows, clean_rows, raw_nulls, clean_nulls)

summary = get_cleaning_summary(df_raw, df)
print(f"Raw rows: {summary[0]}  |  Clean rows: {summary[1]}")
print(f"Raw nulls: {summary[2]}  |  Clean nulls: {summary[3]}")

> 🎯 **Try Out**
> In the blank cell below, write code to create a new column called `salary_band` on the cleaned `df`. Use `pd.cut()` to bin the `salary` column into 3 labeled bands: `"Low"` (bottom third), `"Mid"` (middle third), and `"High"` (top third). Then print the value counts of the new column to see how many employees fall in each band.
>
> *Hint: `pd.cut(df["salary"], bins=3, labels=["Low", "Mid", "High"])` does the binning.*
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!


---

### Challenge 3: Add a Derived Column

Now that `hire_date` is a proper datetime, create a new column called `years_employed`:
1. Use `pd.Timestamp.today()` to get today's date.
2. Subtract `hire_date` from today — this gives a `timedelta` (a duration).
3. Access `.dt.days` and divide by `365` to convert days into years, then round to 1 decimal.
4. Preview `name`, `hire_date`, and `years_employed` for the first 5 rows.

*Major Hint: `(pd.Timestamp.today() - df["hire_date"]).dt.days / 365` gives you the years as a float. You can then call `.round(1)` on it.*

**⚡ How to run the code:**
Write your solution below, then press **Shift + Enter**. *(Solution at the bottom!)*

In [ ]:
# Challenge 3: Write your solution here!

# Step 1: Get today's date

# Step 2 & 3: Compute years_employed and round to 1 decimal

# Step 4: Preview the result


---

# Activity 7: Post-Cleaning Validation

Always verify that your transformations produced the expected results before exporting to S3.

### Step 7.1: Run a Full Validation Check

💡 **What this code does:**
- Checks that no nulls or duplicates remain across the entire DataFrame after all cleaning steps.
- Confirms data types are correct for `salary` (integer) and `hire_date` (datetime).
- Verifies `status` and `department` contain only the expected standardized values.
- Confirms no names still have extra whitespace.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
print("=== Post-Cleaning Validation ===")
print(f"Shape          : {df.shape}")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Duplicates     : {df.duplicated().sum()}")
print(f"Status values  : {sorted(df['status'].unique())}")
print(f"Dept values    : {sorted(df['department'].unique())}")
print(f"Salary range   : {df['salary'].min():,} - {df['salary'].max():,}")
print(f"hire_date type : {df['hire_date'].dtype}")
print(f"Names with spaces: {df['name'].str.startswith(' ').sum() + df['name'].str.endswith(' ').sum()}")

![Images](images/execution24.png)

---

# Activity 8: Export Cleaned Data to S3 as Parquet

Parquet is the industry-standard format for Silver-zone analytics data — it is columnar, compressed, and far faster to query than CSV.

### Step 8.1: Serialize the DataFrame to Parquet In Memory

💡 **What this code does:**
- `io.BytesIO()` creates an in-memory byte buffer — it acts exactly like a file without writing anything to disk.
- `.to_parquet()` writes the cleaned DataFrame into that buffer using Parquet format and Snappy compression.
- Snappy is a fast compression algorithm — it reduces file size significantly without slowing down reads much.
- `.seek(0)` rewinds the buffer pointer back to the start so the upload step can read from the beginning.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Create an in-memory byte buffer — no local file will be written to disk
parquet_buffer = io.BytesIO()

# Write the cleaned DataFrame to Parquet format with Snappy compression
df.to_parquet(parquet_buffer, index=False, engine="pyarrow", compression="snappy")

# Rewind the buffer to the start before reading/uploading
parquet_buffer.seek(0)

print(f"DataFrame serialized to Parquet in memory")
print(f"   Buffer size: {parquet_buffer.getbuffer().nbytes / 1024:.1f} KB")

![Images](images/execution25.png)


### Step 8.2: Upload the Parquet File to S3

💡 **What this code does:**
- `s3.put_object()` uploads the Parquet bytes directly to S3 at the `cleaned/` path — the Silver zone.
- `Body=parquet_buffer.getvalue()` reads all bytes from the buffer and sends them as the file content to S3.
- This is the clean, production-ready upload pattern — no intermediate temp files on disk.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Upload the Parquet bytes to the S3 cleaned/ prefix
s3.put_object(
    Bucket=BUCKET_NAME,
    Key=CLEANED_KEY,
    Body=parquet_buffer.getvalue()
)

print(f"Cleaned Parquet uploaded successfully:")
print(f"   s3://{BUCKET_NAME}/{CLEANED_KEY}")

![Images](images/execution26.png)

> 💡 **TODO**
> Complete the function below to list all objects in the `cleaned/` prefix of your S3 bucket and return their keys and sizes.
>
> Please ensure you complete and execute the code cell below, as your results will be validated and automatically reflected in your **Practice Progress Summary** to track your learning milestones.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Complete the code with the `# TODO` comments. Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.


In [ ]:
def list_cleaned_objects(s3_client, bucket_name, prefix):
    result = None

    # code starts here

    # TODO: Use s3_client to list objects in the bucket under the given prefix
    # TODO: Build a list of (key, size) tuples from the response and store in 'result'

    # code ends here

    return result

objects = list_cleaned_objects(s3, BUCKET_NAME, "cleaned/")
print("Cleaned S3 Objects:")
for key, size in objects:
    print(f"  {key}  ({size / 1024:.1f} KB)")

> 🎯 **Try Out**
> In the blank cell below, use `s3.list_objects_v2()` to list all objects in the `raw/` prefix of your bucket and print each key. Then compare the file sizes of the raw CSV and the cleaned Parquet (in KB) by computing `size / 1024`. Which is smaller, and why?
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!


---

# Activity 9: Verify S3 Upload via AWS Console

Verify that the wrangling script successfully placed the Parquet file into S3.

1. Return to the **AWS Management Console** and navigate to **S3**.
2. Click on your bucket: `data-wrangling-lab-<USER_INPUT>`.
3. Click on the `cleaned/` folder.
![Images](images/aws_s31.png)
4. Confirm `employees_cleaned.parquet` is listed.

![Images](images/aws_s32.png)

5. Click the file and open the **Properties** tab — Observe that the Parquet file size is significantly smaller than the original CSV, confirming effective columnar storage and Snappy compression.

![Images](images/aws_s33.png)

**Expected Observations:**
- The file `employees_cleaned.parquet` is visible in the `cleaned/` folder.
- File size is smaller than the original CSV due to compression.
- The upload timestamp matches the time you ran the notebook.

---

# Activity 10: Read Back & Round-Trip Validation

### Step 10.1: Download the Parquet File from S3 and Read It Back

💡 **What this code does:**
- Downloads the Parquet file from S3 and reads it into a new DataFrame `df_check`.
- This **round-trip test** confirms the file was written correctly and that all column types were preserved exactly.
- Parquet is a schema-preserving format — `salary` stays `int64` and `hire_date` stays `datetime64` on read-back, unlike CSV which would reload everything as strings.

> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.

**⚡ How to run the code:**
Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Download the Parquet file from S3
response      = s3.get_object(Bucket=BUCKET_NAME, Key=CLEANED_KEY)
parquet_bytes = response["Body"].read()

# Read back into a DataFrame using an in-memory buffer
df_check = pd.read_parquet(io.BytesIO(parquet_bytes), engine="pyarrow")

print("=== Round-Trip Verification ===")
print(f"Shape  : {df_check.shape}")
print(f"\nColumn types:")
print(df_check.dtypes)
print()
df_check.head()

![Images](images/execution27.png)

**Expected Observations:**
- Shape remains (1000, 6)
- salary is int64 and hire_date is datetime64[us]
- No missing values or duplicates

---

### Challenge 4: Spot-Check the Data Quality

Use `df_check` (the round-tripped DataFrame) to run a final quality report:
1. Print the **total null count** across all columns using `.isnull().sum().sum()`.
2. Print the **salary range** (min to max) using Pandas `.min()` and `.max()`.
3. Use `df_check.groupby("department")["salary"].mean().round(0)` to compute the **average salary per department** — then sort from highest to lowest.

*Major Hint: Chain `.sort_values(ascending=False)` after the `.mean().round(0)` to sort the result.*

**⚡ How to run the code:**
Write your solution below, then press **Shift + Enter**. *(Solution at the bottom!)*

In [ ]:
# Challenge 4: Write your solution here!

# Step 1: Total null count across all columns

# Step 2: Salary range (min to max)

# Step 3: Average salary per department, sorted highest to lowest


---

## **Please ensure that the file is saved before exiting, as this is required for validation**

## 🎓 Conclusion

This lab took you through a complete, professional-grade **Bronze to Silver** data wrangling pipeline. You started with a raw, messy CSV file containing over 1,000 employee records and transformed it — step by step — into a clean, type-correct, compressed Parquet dataset stored in Amazon S3.

You used **Pandas** for DataFrame profiling, deduplication, type casting, and standardization. You used **NumPy** for vectorized statistical analysis and IQR-based outlier detection. And you used **Boto3** to automate every S3 interaction — upload, download, and list — without touching the AWS Console once the pipeline was running.

This is the exact pattern used in enterprise data lakes every day. You are now ready to build on it with more complex transformations and larger datasets.

---

## 📋 Final Deliverables Checklist

- [ ] AWS CLI configured with `aws configure`; `aws sts get-caller-identity` passes
- [ ] S3 bucket created with `raw/` and `cleaned/` folders
- [ ] `employees_raw.csv` uploaded to `s3://your-bucket/raw/`
- [ ] All notebook cells complete with outputs visible
- [ ] Profiling output shows identified issues (nulls, duplicates, casing, whitespace)
- [ ] NumPy IQR outlier check identifies salary outliers
- [ ] All cleaning transformations applied and post-cleaning validation passes
- [ ] `employees_cleaned.parquet` uploaded to `s3://your-bucket/cleaned/`
- [ ] Read-back verification confirms correct shape and data types

---

## 🔑 Key Concepts Recap

| Concept | What You Did |
|---|---|
| **Bronze to Silver** | Loaded raw CSV from S3, cleaned it, wrote Parquet back to S3 — a classic medallion pipeline step |
| **Data Profiling** | Used `.isnull()`, `.duplicated()`, `.unique()`, `.dtypes` to understand quality issues before touching data |
| **NumPy IQR** | Extracted salary as an array; used `np.percentile` to compute IQR bounds and flag statistical outliers |
| **Pandas Cleaning** | Dropped duplicates, standardised casing, stripped whitespace, filled nulls, cast types |
| **Parquet + Snappy** | Serialised cleaned data to a columnar, compressed format — the Silver-zone standard |
| **Boto3 S3** | Used `upload_file`, `get_object`, and `put_object` for all S3 interactions |
| **Schema-on-write** | Parquet preserves column types — downstream query engines (Athena, Spark) read the exact schema you set |

---
